# Descrição do notebook

- Objetivo
    - Preparar e validar dados meteorológicos: filtrar todos os CSVs em data/raw pela coluna `UF == 'RJ'`, salvar os resultados em `data/staged` e verificar os arquivos gerados.

- Principais passos
    
    1. Configuração de diretórios: `DATA_DIR` (`data`), `RAW_DIR`, `STAGED_DIR`, `PROCESSED_DIR`.
    2. `ensure_dirs()` — cria/garante existência das pastas.
    3. Loop sobre `RAW_DIR.glob("*.csv")`:
         - lê cada CSV com pandas;
         - verifica existência da coluna `UF`;
         - filtra linhas com `UF == 'RJ'`;
         - grava CSV filtrado em `STAGED_DIR` (mesmo nome de arquivo);
         - registra erros/avisos/informações no logger.
    4. `check_all_files_staged()` — lista arquivos em `STAGED_DIR` e lança erro se vazio.
    5. `check_size_files_staged()` — obtém tamanhos (MB) dos arquivos em `STAGED_DIR` e faz log.

- Estado atual (variáveis relevantes)
    - df: dataframe original com 5.173.776 linhas e 27 colunas (tipos mistos).
    - filtered: dataframe filtrado (somente RJ) com 228.384 linhas.
    - filtered_files: lista de arquivos em `data/staged` (7 arquivos: 2018, 2019, 2020, 2021, 2022, 2023, 2024).
    - logger configurado em nível INFO.

- Observações
    - O pipeline trata falhas de leitura/gravação e ausência da coluna `UF` via logs.
    - Próximos passos típicos: limpeza/normalização de colunas (conversão de tipos numéricos, tratamento de vírgulas/NaN), agregações por estação/tempo, e mover resultados para `PROCESSED_DIR`.

In [ ]:
import os
import logging
from pathlib import Path
import pandas as pd

In [3]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [4]:
# Config
DATA_DIR = Path(os.environ.get("DATA_DIR", "data"))
DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR = DATA_DIR / "raw"
STAGED_DIR = DATA_DIR / "staged"
PROCESSED_DIR = DATA_DIR / "processed"

In [ ]:
def ensure_dirs():
    logger.info("Ensuring data directories exist.")
    if not RAW_DIR.exists():
        RAW_DIR.mkdir(parents=True, exist_ok=True)
        logger.info(f"Created RAW_DIR at {RAW_DIR}")
    if not STAGED_DIR.exists():
        STAGED_DIR.mkdir(parents=True, exist_ok=True)
        logger.info(f"Created STAGED_DIR at {STAGED_DIR}")
    if not PROCESSED_DIR.exists():
        PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
        logger.info(f"Created PROCESSED_DIR at {PROCESSED_DIR}")

In [6]:
ensure_dirs()

INFO:__main__:Ensuring data directories exist.
INFO:__main__:Created RAW_DIR at data/raw
INFO:__main__:Created STAGED_DIR at data/staged
INFO:__main__:Created PROCESSED_DIR at data/processed


In [ ]:
# Filtrar todos os CSVs em RAW_DIR por UF == 'RJ' e salvar em STAGED_DIR
for raw_file in RAW_DIR.glob("*.csv"):
    logger.info(f"Processing file {raw_file}")
    try:
        df = pd.read_csv(raw_file)
    except Exception as e:
        logger.error(f"Falha ao ler {raw_file}: {e}")
        continue

    if 'UF' not in df.columns:
        logger.warning(f"Ignorando {raw_file}: coluna 'UF' não encontrada")
        continue

    filtered = df[df['UF'] == 'RJ']
    out_file = STAGED_DIR / raw_file.name
    try:
        filtered.to_csv(out_file, index=False)
        logger.info(f"Gravado {len(filtered)} linhas em {out_file}")
    except Exception as e:
        logger.error(f"Falha ao gravar {out_file}: {e}")

/tmp/ipykernel_21676/451862745.py:4: DtypeWarning: Columns (2: PRECIPITAÇÃO TOTAL, HORÁRIO (mm), 15: UMIDADE RELATIVA DO AR, HORARIA (%), 23: LATITUDE) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(raw_file)
INFO:__main__:Gravado 227760 linhas em data/staged/2021.csv
/tmp/ipykernel_21676/451862745.py:4: DtypeWarning: Columns (23: LATITUDE) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(raw_file)
INFO:__main__:Gravado 227760 linhas em data/staged/2019.csv
/tmp/ipykernel_21676/451862745.py:4: DtypeWarning: Columns (23: LATITUDE) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(raw_file)
INFO:__main__:Gravado 228384 linhas em data/staged/2020.csv
/tmp/ipykernel_21676/451862745.py:4: DtypeWarning: Columns (23: LATITUDE) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(raw_file)
INFO:__main__:Gravado 227760 linha

In [ ]:
def check_all_files_staged():
    logger.info(f"Checking all files in {STAGED_DIR}")
    files = list(STAGED_DIR.glob("*"))
    logger.info(f"Listing files in {STAGED_DIR}: {files}")
    if not files:
        logger.error(f"No files found in {STAGED_DIR}")
        raise FileNotFoundError(f"No files found in {STAGED_DIR}")
    for file in files:
        logger.info(f"Found file: {file}")
    return files

In [ ]:
def check_size_files_staged():
    logger.info(f"Checking sizes of all files in {STAGED_DIR}")
    files = check_all_files_staged()
    sizes_mb = {}
    if not files:
        logger.error(f"No files found in {STAGED_DIR}")
        raise FileNotFoundError(f"No files found in {STAGED_DIR}")
    logger.info(f"Found {len(files)} files in {STAGED_DIR}")
    for file in files:
        logger.info(f"Checking size for file: {file}")
        size_bytes = file.stat().st_size
        size_mb = size_bytes / (1024 ** 2)
        logger.info(f"File: {file}, Size: {size_mb:.2f} MB")
        sizes_mb[file] = size_mb
        logger.info(f"Recorded size for file: {file}, Size: {size_mb:.2f} MB")
        logger.info(f"Finished checking size for file: {file}")
    logger.info(f"Finished checking sizes for all files in {STAGED_DIR}")
    return sizes_mb

In [10]:
check_size_files_staged()

INFO:__main__:Found file: data/staged/2021.csv
INFO:__main__:Found file: data/staged/2019.csv
INFO:__main__:Found file: data/staged/2020.csv
INFO:__main__:Found file: data/staged/2023.csv
INFO:__main__:Found file: data/staged/2018.csv
INFO:__main__:Found file: data/staged/2024.csv
INFO:__main__:Found file: data/staged/2022.csv
INFO:__main__:File: data/staged/2021.csv, Size: 34.90 MB
INFO:__main__:File: data/staged/2019.csv, Size: 36.71 MB
INFO:__main__:File: data/staged/2020.csv, Size: 34.24 MB
INFO:__main__:File: data/staged/2023.csv, Size: 36.57 MB
INFO:__main__:File: data/staged/2018.csv, Size: 31.79 MB
INFO:__main__:File: data/staged/2024.csv, Size: 17.45 MB
INFO:__main__:File: data/staged/2022.csv, Size: 35.29 MB


{PosixPath('data/staged/2021.csv'): 34.90126609802246,
 PosixPath('data/staged/2019.csv'): 36.71233081817627,
 PosixPath('data/staged/2020.csv'): 34.23577308654785,
 PosixPath('data/staged/2023.csv'): 36.566046714782715,
 PosixPath('data/staged/2018.csv'): 31.791444778442383,
 PosixPath('data/staged/2024.csv'): 17.450071334838867,
 PosixPath('data/staged/2022.csv'): 35.29421615600586}